In [35]:
## LIBRARY
from pathlib import Path
import glob
import json
import numpy as np
from skimage.io import imsave
from nd2 import ND2File
import warnings
import pandas as pd
from sklearn.cluster import DBSCAN
from skimage.draw import ellipse_perimeter
from skimage.exposure import rescale_intensity
from skimage.color import gray2rgb
from skimage.restoration import rolling_ball
from matplotlib import colors as mcolors
from scipy.optimize import OptimizeWarning
from skimage import io

from calmutils.localization import refine_point_lsq, detect_dog
from calmutils.localization.util import sigma_to_full_width_at_quantile, full_width_at_quantile_to_sigma

from skimage.feature import blob_dog
from concurrent.futures import ThreadPoolExecutor


In [90]:
# path containing files to visualize
in_path = './20240819_spinning_disk_tetraspeck_beads/corrected_images_tetraspeck_561'

# subdirectory containing input data
# leave empty ('') if image files are directly in in_path
in_subdirectory = ''

# default: put results in subdirectory called 'spot-detection'
out_subdirectory = 'Results_corr_ch561'

# which channels to include
channels_to_include = ['488-CSU-W1', '561-CSU-W1', '640-CSU-W1']

# DoG threshold for all channels
thresholds_dog = 0.02

# Alternative: threshold_log can be a dictionary containing a separate threshold for each channel
thresholds_dog = {
    '488-CSU-W1': 20, #this - how to choose threshold? Experimenting, basically
    '561-CSU-W1': 15,
    '640-CSU-W1': 35
}
threshold_dog_relative = False
thresholds_intensity = None

#can ignore this for now:
do_background_subtraction = False
background_subtraction_radius = 5

# refine points via Gaussian fit?
do_gaussian_fit = True

# expected size (zyx, in microns)
expected_size = [0.7, 0.35, 0.35]

# how many peaks to detect at max
# NOTE: this is mainly a fail-safe to prevent needlessly long computations when thresholds are set incorrectly
max_num_peaks = 2000

# maximum log2 fold deviation from expected size
# can be used to filter out large objects, e.g. 1: fitted gaussian has to be within 0.5 - 2x expected size
# set to None to skip
max_log2_deviation_from_expected_size = 1

# cluster rejection via DBSCAN clustering
# spots that lie in clusters in which at least cluster_reject_n_spots spots lie within cluster_reject_distance are ignored
# cluster_reject_distance is in units of expected size (i.e. 5.0: are considered to cluster if their distance is < 5*expected_size)
cluster_reject_distance = 5.0
cluster_reject_n_spots = 5

save_visualization = True

## ALSO SHOULD SKIP FOR NOW (needlessly complicated)
# how many images to process in parallel
num_threads = 16

## added parameters?
channel_name = '561-CSU-W1'
pixel_size = tuple([0.3, 0.13, 0.13])


In [91]:
### PATHS AND PARAMETERS

# make Paths
in_path = Path(in_path)
out_path = in_path / out_subdirectory

# make dict with same threshold for all channels
if not isinstance(thresholds_dog, dict):
    thresholds_dog = {channel: thresholds_dog for channel in channels_to_include}
if not isinstance(thresholds_intensity, dict):
    thresholds_intensity = {channel: thresholds_intensity for channel in channels_to_include}

expected_size = np.array(expected_size)

parameter_log = {
    'in_path': str(in_path),
    'in_subdirectory': in_subdirectory,
    'channels_to_include': channels_to_include,
    'thresholds_dog': thresholds_dog,
    'thresholds_intensity': thresholds_intensity,
    'do_gaussian_fit': do_gaussian_fit,
    'expected_size': list(expected_size),
    'cluster_reject_distance': cluster_reject_distance,
    'cluster_reject_n_spots': cluster_reject_n_spots,
    'max_log2_deviation_from_expected_size': max_log2_deviation_from_expected_size,
    'max_num_peaks': max_num_peaks,
    'do_background_subtraction': do_background_subtraction,
    'background_subtraction_radius': background_subtraction_radius,
    'threshold_dog_relative': threshold_dog_relative
}

# get all nd2 files in in_path
in_files = sorted((Path(in_path) / in_subdirectory).glob('*.tif'))

# show for verification
in_files

[WindowsPath('20240819_spinning_disk_tetraspeck_beads/corrected_images_tetraspeck_561/20240819-tetraspack-001_corr_ch1.tif'),
 WindowsPath('20240819_spinning_disk_tetraspeck_beads/corrected_images_tetraspeck_561/20240819-tetraspack-002_corr_ch1.tif'),
 WindowsPath('20240819_spinning_disk_tetraspeck_beads/corrected_images_tetraspeck_561/20240819-tetraspack-003_corr_ch1.tif'),
 WindowsPath('20240819_spinning_disk_tetraspeck_beads/corrected_images_tetraspeck_561/20240819-tetraspack-004_corr_ch1.tif'),
 WindowsPath('20240819_spinning_disk_tetraspeck_beads/corrected_images_tetraspeck_561/20240819-tetraspack-005_corr_ch1.tif'),
 WindowsPath('20240819_spinning_disk_tetraspeck_beads/corrected_images_tetraspeck_561/20240819-tetraspack-006_corr_ch1.tif'),
 WindowsPath('20240819_spinning_disk_tetraspeck_beads/corrected_images_tetraspeck_561/20240819-tetraspack-007_corr_ch1.tif'),
 WindowsPath('20240819_spinning_disk_tetraspeck_beads/corrected_images_tetraspeck_561/20240819-tetraspack-008_corr_ch1

In [92]:
### functions
def refine_points(image, points, sigma_expected=None, max_log2_deviation_from_expected_size=None):

    points_refined = []
    sigmas_refined = []
    minmax_refined = []

    # if expected size is given, cut until expected full-with at 5% intenisty of gaussian
    cutregion = np.round(sigma_to_full_width_at_quantile(sigma_expected, 0.05) / 2).astype(int) if sigma_expected is not None else None

    for blob in points:
        # do Gaussian fit, ignore warnings about failed optimization -> we will skip those blobs
        with warnings.catch_warnings():
            warnings.simplefilter('ignore', (OptimizeWarning, RuntimeWarning))
            pos_refined, fit = refine_point_lsq(image, blob, cutregion)
        # skip if fit not possible or negative sigma
        if fit is None:
            continue
        fit, _ = fit
        if np.any(fit[-3:] < 0) or np.any(np.isnan(fit)):
            continue
        sigma = fit[-3:]

        # discard spot if the sigma deviates too much from expected size
        # we calculate the mean absolute log2 of ratio expected/fit and discard if bigger than our threshold
        if sigma_expected is not None and max_log2_deviation_from_expected_size is not None:
            if np.abs(np.log2(np.array(sigma_expected) / np.array(sigma))).mean() > max_log2_deviation_from_expected_size:
                continue

        points_refined.append(pos_refined)
        sigmas_refined.append(sigma)
        minmax_refined.append(fit[:2])

    points_refined = np.array(points_refined)
    return points_refined, np.array(sigmas_refined), np.array(minmax_refined)

def filter_clustering(blobs, pixel_size, expected_size, cluster_reject_distance, cluster_reject_n_spots):
    # drop sigma columns produced by blob_log/blob_dog
    blobs_just_coords = blobs * pixel_size / expected_size
    # spots that receive class -1 in DBSCAN := not in cluster
    # NOTE: explicitly setting algorithm='kd_tree' was necessary to avoid issues in multithreaded processing of files
    single_spot_idx = DBSCAN(cluster_reject_distance, min_samples=cluster_reject_n_spots, algorithm='kd_tree').fit_predict(blobs_just_coords) == -1
    return blobs[single_spot_idx], single_spot_idx

def get_spot_visualization_projection(img, blobs, sigmas):

    # color to plot ellipse in
    ellipse_color = np.array(mcolors.hex2color(mcolors.XKCD_COLORS['xkcd:sea green']))
    # how much to expand the ellipse with radius = full width at tenth maximum
    ellipse_expansion_factor = 4
    # ellipse line width
    ellipse_line_width = 3

    img_projected = img.max(axis=0)
    proj_rgb = gray2rgb(rescale_intensity(img_projected, in_range=tuple(np.quantile(img_projected, (0.02, 0.9999))), out_range='float32'))

    for blob, sigma in zip(blobs, sigmas):
        
        # get position and sigma of blob in yx
        yx = blob[1:].astype(int)
        sy_sx = sigma[1:]
        # to radius of ellipse (based on full width at tenth maximum times expansion factor)
        ry_rx = (sigma_to_full_width_at_quantile(sy_sx, 0.1) / 2 * ellipse_expansion_factor).astype(int)
        
        # line width: draw single pixel ellipse at radius + 0, +1, ...
        for i in range(ellipse_line_width):
            proj_rgb[tuple(ellipse_perimeter(*yx, *(ry_rx+i), shape=proj_rgb.shape))] = ellipse_color

    proj_rgb = (proj_rgb * 255).astype(np.uint8)

    return proj_rgb

def blobs_to_df(blobs_i, file_path, sigmas, minmax): #used to have: position (3rd place)

    #pixel size is a parameter...

    df = pd.DataFrame()
    for channel_name, blobs_ii in blobs_i.items(): 

        # NOTE: reshape to prevent errors on empty results ((0,3)-array)
        # should also result in column names not being dropped
        blobs_ii = blobs_ii.reshape((-1, 3))

        df_i = pd.DataFrame({
            'spot_idx': np.arange(len(blobs_ii), dtype=int),
            'image_file': file_path,
            #'position_idx': position,
            'channel': channel_name,
            **dict(zip('zyx', blobs_ii.T)), 
            **dict(zip(['z_micron', 'y_micron', 'x_micron'], (blobs_ii * pixel_size).T))
            })
        
        sigmas_i = sigmas.get(channel_name, None)
        minmax_i = minmax.get(channel_name, None)

        # reshape to keep track of dimensionality even for empty results, see above
        if sigmas_i is not None:
            sigmas_i = sigmas_i.reshape((-1, 3))
        if minmax_i is not None:
            minmax_i = minmax_i.reshape((-1, 2))
        
        if sigmas_i is not None:
            for col_name, vals in zip(['sigma_z', 'sigma_y', 'simga_x'], sigmas_i.T):
                df_i[col_name] = vals
        if sigmas_i is not None:
            for col_name, vals in zip(['sigma_z_micron', 'sigma_y_micron', 'simga_x_micron'], (sigmas_i * pixel_size).T):
                df_i[col_name] = vals
        if minmax_i is not None:
            for col_name, vals in zip(['gauss_fit_min', 'gauss_fit_height'], minmax_i.T):
                df_i[col_name] = vals

        df = pd.concat([df, df_i], ignore_index=True)
    
    return df

def imsave_nowarnings(file, img, **kwargs):
    # catch low contrast warning
    with warnings.catch_warnings():
        warnings.simplefilter('ignore', UserWarning)
        imsave(file, img, **kwargs)

In [93]:
### Repurposing to tiff

### Loading:
blobs_list = []
visualization_projections_list = []
sigmas_list = []
minmaxs_list = []

for file in in_files:
    image = io.imread(file).astype(float)
    kwargs = parameter_log | {'in_file': file, 'do_visualization': save_visualization} #used to have: 'position_idx': position_idx,

    # sigma for expected size in pixels
    sigma_expected = full_width_at_quantile_to_sigma(expected_size) / pixel_size

    # results will be dicts mapping channel names to result values
    blobs = {}
    sigmas = {}
    minmax = {}
    projections = {}
    
    # I can ignorme blobs_i?
    blobs_i = detect_dog(image, 
            threshold=thresholds_dog[channel_name], threshold_intensity=thresholds_intensity[channel_name],
            sigma=sigma_expected, max_num_peaks=max_num_peaks, threshold_relative=threshold_dog_relative)
    # we have only one sigma, repeat for each detection
    sigmas_i = np.tile(sigma_expected, len(blobs_i)).reshape((-1, image.ndim))


    ## essentially curve fitting - fitting curve to a signal (see co-calc spot_detection_curve_fitting)
    if do_gaussian_fit:
        blobs_i, sigmas_i, minmax_i = refine_points(image, blobs_i, sigma_expected, max_log2_deviation_from_expected_size)

    if len(blobs_i) > cluster_reject_n_spots:
        blobs_i, single_spot_idx = filter_clustering(blobs_i, pixel_size, expected_size, cluster_reject_distance, cluster_reject_n_spots)
        sigmas_i = sigmas_i[single_spot_idx]
        if do_gaussian_fit:
            minmax_i = minmax_i[single_spot_idx]
        
    blobs[channel_name] = blobs_i

    if do_gaussian_fit:
        sigmas[channel_name] = sigmas_i
        minmax[channel_name] = minmax_i

    # generate projection with detections visualized
    visualization_projection = get_spot_visualization_projection(image, blobs_i, sigmas_i)
    projections[channel_name] = visualization_projection

    #save results (dicts) into lists
    blobs_list.append(blobs)
    visualization_projections_list.append(projections)
    sigmas_list.append(sigmas)
    minmaxs_list.append(minmax)

## paths:
if not out_path.exists():
    out_path.mkdir(parents=True)

visualization_path = out_path / 'quick_result_visualization'
if save_visualization and not visualization_path.exists():
    visualization_path.mkdir(parents=True)

i = 0
for file in in_files:
    df = blobs_to_df(blobs_list[i], file, sigmas_list[i], minmaxs_list[i])
    out_file = out_path / (file.stem + '_spot-detection.csv')
    df.to_csv(out_file, index=None)

    if save_visualization:
        for channel_name, visualization_projection in visualization_projections_list[i].items():
            # make filepath for output
            outfile_visualization = visualization_path / f'_{channel_name}_spot-detection_{i}.png'      
            imsave_nowarnings(str(outfile_visualization), visualization_projection)
    i += 1

# remaining:
with open(out_path / 'spot_detection_parameters.json', 'w') as fd:
    json.dump(parameter_log, fd, indent=1)
